In [2]:
import torch
import math
import torch.nn as nn

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [4]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous()
        out = out.view(batch_size, seq_len, self.d_model)

        return self.fc_out(out)


In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out = self.attention(x)
        x = self.norm1(x + attn_out)      # Add & Norm

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)       # Add & Norm

        return x


In [8]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)

        for layer in self.layers:
            x = layer(x)

        return self.fc_out(x)


In [13]:
# Suppose vocabulary: {"I":0, "love":1, "AI":2, "<pad>":3}
token_ids = [0, 1, 2]

# Convert to tensor and batch dimension
input_tokens = torch.tensor([token_ids])  # shape: [batch_size=1, seq_len=3]


In [14]:
# Hyperparameters
vocab_size = 4
d_model = 8
num_heads =2
d_ff = 32
num_layers = 1

model = Transformer(vocab_size, d_model, num_heads, d_ff, num_layers)

# Example input (batch_size=1, sequence_length=5)
# input_tokens = torch.tensor([[10, 20, 30, 40, 50]])

output = model(input_tokens)

print(output.shape)


torch.Size([1, 3, 4])


In [15]:
probabilities = torch.softmax(output, dim=-1)
print(probabilities)

tensor([[[0.2167, 0.2035, 0.4341, 0.1457],
         [0.1407, 0.5880, 0.1218, 0.1496],
         [0.4391, 0.2844, 0.1282, 0.1482]]], grad_fn=<SoftmaxBackward0>)


In [16]:
predicted_tokens = torch.argmax(probabilities, dim=-1)
print("Predicted token IDs:", predicted_tokens)


Predicted token IDs: tensor([[2, 1, 0]])


In [17]:
pip install pdfplumber


   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/6.6 MB 5.6 MB/s eta 0:00:02
   --------- ------------------------------ 1.6/6.6 MB 6.5 MB/s eta 0:00:01
   ----------------------- ---------------- 3.9/6.6 MB 6.9 MB/s eta 0:00:01
   ------------------------------- -------- 5.2/6.6 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------  6.6/6.6 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 6.1 MB/s  0:00:01
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   -------------------- ------------------- 1.8/3.5 MB 8.4 MB/s eta 0:00:01
   ----------------------------------- ---- 3.1/3.5 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 7.2 MB/s  0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.1 MB 4.2 MB/s eta 0:00:01
   -------------------- ---------------

In [18]:
# -----------------------------
# 1️⃣ Required Libraries
# # -----------------------------
# !pip install torch torchvision torchaudio transformers pdfplumber sentencepiece --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pdfplumber
import re
import random

# -----------------------------
# 2️⃣ Load PDF and preprocess
# -----------------------------
pdf_path = "transformers.pdf"  # path to your PDF

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + " "
    return text

raw_text = extract_text_from_pdf(pdf_path)
# basic cleaning
raw_text = re.sub(r'\s+', ' ', raw_text)

# split text into chunks (like context paragraphs)
chunks = raw_text.split(". ")
chunks = [c.strip() for c in chunks if len(c) > 20]

# -----------------------------
# 3️⃣ Create simple QA pairs (context -> question)
# -----------------------------
# For demo, we generate simple questions (like: "What is in context?")
qa_pairs = []
for c in chunks:
    # very naive question: "What does the text say?"
    question = "What does the text say?"
    answer = c
    qa_pairs.append((question, answer))

# -----------------------------
# 4️⃣ Simple tokenizer (character-level for demo)
# -----------------------------
all_text = " ".join([q + " " + a for q,a in qa_pairs])
chars = list(set(all_text))
char2idx = {ch:i for i,ch in enumerate(chars)}
idx2char = {i:ch for ch,i in char2idx.items()}
vocab_size = len(char2idx)

def text_to_seq(text):
    return [char2idx[ch] for ch in text if ch in char2idx]

def seq_to_text(seq):
    return "".join([idx2char[i] for i in seq])

# -----------------------------
# 5️⃣ Dataset
# -----------------------------
class QADataset(Dataset):
    def __init__(self, pairs, max_len=200):
        self.pairs = pairs
        self.max_len = max_len
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        q, a = self.pairs[idx]
        q_seq = text_to_seq(q)
        a_seq = text_to_seq(a)
        
        # pad sequences
        q_seq = q_seq[:self.max_len] + [0]*(self.max_len - len(q_seq))
        a_seq = a_seq[:self.max_len] + [0]*(self.max_len - len(a_seq))
        return torch.tensor(q_seq), torch.tensor(a_seq)

dataset = QADataset(qa_pairs)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# 6️⃣ Simple Encoder-Decoder Transformer
# -----------------------------
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Parameter(torch.randn(200, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, src, tgt):
        # src/tgt shape: [batch, seq_len]
        src_emb = self.embedding(src) + self.pos_encoder[:src.size(1), :]
        tgt_emb = self.embedding(tgt) + self.pos_encoder[:tgt.size(1), :]
        
        src_emb = src_emb.transpose(0,1)  # transformer expects [seq, batch, d]
        tgt_emb = tgt_emb.transpose(0,1)
        
        memory = self.encoder(src_emb)
        output = self.decoder(tgt_emb, memory)
        output = self.fc_out(output)
        return output.transpose(0,1)  # [batch, seq, vocab_size]

# -----------------------------
# 7️⃣ Train the model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTransformer(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5  # for demo; increase for better results
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for q_seq, a_seq in dataloader:
        q_seq = q_seq.to(device)
        a_seq = a_seq.to(device)
        
        optimizer.zero_grad()
        output = model(q_seq, a_seq[:,:-1])
        loss = criterion(output.reshape(-1, vocab_size), a_seq[:,1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

# -----------------------------
# 8️⃣ Simple Chatbot Inference
# -----------------------------
def generate_answer(model, question, max_len=200):
    model.eval()
    with torch.no_grad():
        q_seq = text_to_seq(question)
        q_seq = q_seq[:max_len] + [0]*(max_len - len(q_seq))
        q_seq = torch.tensor([q_seq]).to(device)
        
        # Start token for decoder (can be first char of question)
        tgt_seq = torch.tensor([[0]]).to(device)
        output_seq = []
        
        for _ in range(max_len):
            out = model(q_seq, tgt_seq)
            next_char = out[0,-1,:].argmax().item()
            output_seq.append(next_char)
            tgt_seq = torch.cat([tgt_seq, torch.tensor([[next_char]]).to(device)], dim=1)
        return seq_to_text(output_seq)

# Example usage:
question = "What does the text say?"
answer = generate_answer(model, question)
print("Answer:", answer)


c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1, Loss: 1.9497
Epoch 2, Loss: 1.6472
Epoch 3, Loss: 1.5728
Epoch 4, Loss: 1.4980
Epoch 5, Loss: 1.0895
Answer: 𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑𝑑
